# Imports & Data Prep

In [1]:
import pandas as pd
import numpy as np

In [47]:
cols = ['Date','Account Type','Account Name','Account Number','Name','Amount','Category']
excl_cats = ['Credit Card Payment','Internal Transfers']
df = pd.read_csv('transactions.csv')
df = df[~df['Category'].isin(excl_cats)]
df = df[cols]
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Year_Month'] = pd.to_datetime(df['Date'].dt.month.astype(str) + '/' + df['Date'].dt.year.astype(str), format='%m/%Y')
df['Amount'] = df['Amount'].astype(float)
df['transaction_type'] = np.where(df['Amount'] < 0, 'Debit', np.where(df['Category'] == 'Savings Transfer', 'Transfer', np.where(df['Category'] == 'Investment', 'Investment', 'Credit')))
df = df[df['Category'] + df['transaction_type'] != 'IgnoreCredit'].copy()
df['Amount'] = df['Amount'].abs()

In [48]:
df_dedup = df.drop_duplicates(subset=['Name','Date','Amount'])

# Net Income Over Time

In [49]:
import plotly.graph_objects as go
import numpy as np

df_dedup['Year_Month'] = pd.to_datetime(df_dedup['Year_Month'])
df = df_dedup[(df_dedup['Date'] > '01/01/2023') & (df_dedup['Date'] < '05/01/2026')].sort_values(['Year','Month'])
df = df[['Year_Month','Amount','transaction_type']].groupby(['Year_Month','transaction_type'], as_index=False).sum()
pivoted = df.pivot(index='Year_Month', columns='transaction_type', values='Amount')
pivoted['Net Income'] = pivoted['Debit'] - pivoted['Credit']
pivoted = pivoted.reset_index()[['Year_Month','Net Income']]

x = pivoted['Year_Month'].values
y = pivoted['Net Income'].values

# Interpolate zero crossings so color transitions happen exactly at y=0
new_x, new_y = [], []
for i in range(len(y)):
    new_x.append(x[i])
    new_y.append(y[i])
    if i < len(y) - 1 and y[i] * y[i + 1] < 0:
        t = float(y[i]) / float(y[i] - y[i + 1])
        new_x.append(x[i] + (x[i + 1] - x[i]) * t)
        new_y.append(0.0)

new_x = np.array(new_x)
new_y = np.array(new_y)

y_pos = np.where(new_y >= 0, new_y, 0)
y_neg = np.where(new_y <= 0, new_y, 0)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=new_x, y=y_pos,
    fill='tozeroy', mode='lines',
    line=dict(color='green', width=1.5),
    fillcolor='rgba(0,180,0,0.3)',
    name='Surplus'
))
fig.add_trace(go.Scatter(
    x=new_x, y=y_neg,
    fill='tozeroy', mode='lines',
    line=dict(color='red', width=1.5),
    fillcolor='rgba(200,0,0,0.3)',
    name='Deficit'
))
fig.update_layout(title='Net Income Over Time', xaxis_title='Month', yaxis_title='Net Income')
fig.show()

/tmp/ipykernel_901614/2569771077.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



# Net Income Distribution

In [52]:
from scipy import stats
from plotly.subplots import make_subplots

net = pivoted['Net Income'].dropna().values

# Signed log1p transform: preserves sign for negative values, compresses scale
transformed = np.sign(net) * np.log1p(np.abs(net))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Net Income (Raw)', 'Net Income (Signed Log Transform)'))

fig.add_trace(go.Histogram(x=net, nbinsx=20, marker_color='steelblue', name='Raw'), row=1, col=1)
fig.add_trace(go.Histogram(x=transformed, nbinsx=20, marker_color='mediumpurple', name='Log-transformed'), row=1, col=2)

# Fit normal to log-transformed data and overlay PDF
mu, sigma = stats.norm.fit(transformed)

x_fit = np.linspace(transformed.min(), transformed.max(), 300)
bin_width = (transformed.max() - transformed.min()) / 20
pdf_scaled = stats.norm.pdf(x_fit, mu, sigma) * len(transformed) * bin_width
fig.add_trace(
    go.Scatter(x=x_fit, y=pdf_scaled, mode='lines',
               line=dict(color='red', width=2),
               name=f'Normal fit (μ={mu:.2f}, σ={sigma:.2f})'),
    row=1, col=2
)

fig.update_layout(title='Net Income Distribution', showlegend=True)
fig.show()

# Bootstrap Simulation of Mortgage Payment Impact on Net Income

In [53]:
rng = np.random.default_rng(42)
net = pivoted['Net Income'].dropna().values

mortgage_costs = np.arange(1600, 5100, 100)
n_sims = 1000
n_months = 24

# Each simulation: sample 24 months from the empirical distribution, sum, subtract total mortgage cost
# Shape: (n_sims, n_months) → sum across months → subtract cost * n_months
sim_results = {
    cost: rng.choice(net, size=(n_sims, n_months), replace=True).sum(axis=1) - cost * n_months
    for cost in mortgage_costs
}

prob_positive = [np.mean(sim_results[c] > 0) for c in mortgage_costs]
mean_net      = [np.mean(sim_results[c]) for c in mortgage_costs]
p10           = [np.percentile(sim_results[c], 10) for c in mortgage_costs]
p90           = [np.percentile(sim_results[c], 90) for c in mortgage_costs]

# Find mortgage cost closest to 50% probability crossover
prob_arr = np.array(prob_positive)
cross_idx = np.argmin(np.abs(prob_arr - 0.5))
crossover_cost = mortgage_costs[cross_idx]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'Probability of Positive 24-Month Net Income',
        '24-Month Cumulative Net Income After Mortgage'
    )
)

fig.add_trace(go.Scatter(
    x=mortgage_costs, y=prob_positive,
    mode='lines+markers',
    line=dict(color='steelblue', width=2),
    marker=dict(size=5),
    name='P(cumulative net > 0)'
), row=1, col=1)
fig.add_hline(y=0.5, line_dash='dash', line_color='red',
              annotation_text=f'50% ≈ ${crossover_cost:,}', row=1, col=1)

fig.add_trace(go.Scatter(
    x=list(mortgage_costs) + list(mortgage_costs[::-1]),
    y=p90 + p10[::-1],
    fill='toself',
    fillcolor='rgba(70,130,180,0.2)',
    line=dict(color='rgba(0,0,0,0)'),
    name='10th–90th pctile'
), row=1, col=2)
fig.add_trace(go.Scatter(
    x=mortgage_costs, y=mean_net,
    mode='lines',
    line=dict(color='steelblue', width=2),
    name='Mean cumulative net'
), row=1, col=2)
fig.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Break-even', row=1, col=2)

fig.update_xaxes(title_text='Monthly Mortgage Cost ($)', tickprefix='$', tickformat=',')
fig.update_yaxes(title_text='Probability', tickformat='.0%', row=1, col=1)
fig.update_yaxes(title_text='Cumulative Net Income ($)', tickprefix='$', tickformat=',', row=1, col=2)
fig.update_layout(
    title=f'Mortgage Cost Impact — {n_months}-Month Horizon, {n_sims:,} Bootstrap Simulations per Level',
    showlegend=True
)
fig.show()

print(f"\n--- Key thresholds (24-month horizon) ---")
for c in mortgage_costs:
    p = np.mean(sim_results[c] > 0)
    if abs(p - 0.5) < 0.05:
        print(f"  ${c:,}/mo  →  P(positive) = {p:.1%}  |  mean = ${np.mean(sim_results[c]):,.0f}  |  p10 = ${np.percentile(sim_results[c], 10):,.0f}")


--- Key thresholds (24-month horizon) ---
  $3,300/mo  →  P(positive) = 54.2%  |  mean = $4,284  |  p10 = $-20,325
  $3,400/mo  →  P(positive) = 49.9%  |  mean = $1,794  |  p10 = $-22,281
  $3,500/mo  →  P(positive) = 49.2%  |  mean = $608  |  p10 = $-25,066


# Effect of Pay Increases on Mortgage Affordability

In [54]:
from plotly.subplots import make_subplots

rng = np.random.default_rng(42)
net = pivoted['Net Income'].dropna().values

salary_increases = np.arange(1000, 21000, 1000)   # Annual raise amounts ($1K–$20K)
mortgage_costs   = np.arange(1600, 5100, 100)      # xMonthly mortgage ($1,600–$5,000)
n_sims   = 1000
n_months = 12   # 1-year horizon

# Sample base net income once and reuse across all combos for efficiency
base_total = rng.choice(net, size=(n_sims, n_months), replace=True).sum(axis=1)  # shape (n_sims,)

prob_grid = np.zeros((len(salary_increases), len(mortgage_costs)))
mean_grid = np.zeros((len(salary_increases), len(mortgage_costs)))

for i, annual_raise in enumerate(salary_increases):
    monthly_raise = annual_raise / 12
    for j, mort in enumerate(mortgage_costs):
        sim = base_total + monthly_raise * n_months - mort * n_months
        prob_grid[i, j] = np.mean(sim > 0)
        mean_grid[i, j] = np.mean(sim)

salary_labels    = [f'${s:,}' for s in salary_increases]
mortgage_labels  = [f'${m:,}' for m in mortgage_costs]

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        'P(Positive 12-Month Net Income)',
        'Mean 12-Month Cumulative Net Income'
    ),
    horizontal_spacing=0.18
)

fig.add_trace(go.Heatmap(
    x=mortgage_costs, y=salary_increases, z=prob_grid,
    colorscale='RdYlGn', zmin=0, zmax=1,
    colorbar=dict(title='Probability', tickformat='.0%', len=0.85, x=0.44, y=0.5),
    hovertemplate='Mortgage: $%{x:,}/mo<br>Raise: $%{y:,}/yr<br>P(net > 0): %{z:.1%}<extra></extra>'
), row=1, col=1)

fig.add_trace(go.Heatmap(
    x=mortgage_costs, y=salary_increases, z=mean_grid,
    colorscale='RdYlGn',
    colorbar=dict(title='Mean Net ($)', tickprefix='$', tickformat=',', len=0.85, x=1.02, y=0.5),
    hovertemplate='Mortgage: $%{x:,}/mo<br>Raise: $%{y:,}/yr<br>Mean net: $%{z:,.0f}<extra></extra>'
), row=1, col=2)

fig.update_xaxes(title_text='Monthly Mortgage Cost', tickprefix='$', tickformat=',')
fig.update_yaxes(title_text='Annual Salary Raise', tickprefix='$', tickformat=',')
fig.update_layout(
    title=f'Salary Raise vs. Mortgage Cost — 12-Month Horizon, {n_sims:,} Bootstrap Simulations',
    height=520
)
fig.show()

# Print the 50% crossover mortgage cost for each raise level
print("\n--- Break-even mortgage cost per raise level (closest to 50% probability) ---")
for i, sal in enumerate(salary_increases):
    crossover_idx = np.argmin(np.abs(prob_grid[i] - 0.5))
    print(f"  Raise ${sal:>6,}/yr  →  ~${mortgage_costs[crossover_idx]:,}/mo mortgage  "
          f"(P={prob_grid[i, crossover_idx]:.0%}, mean net = ${mean_grid[i, crossover_idx]:,.0f})")


--- Break-even mortgage cost per raise level (closest to 50% probability) ---
  Raise $ 1,000/yr  →  ~$3,400/mo mortgage  (P=49%, mean net = $2,005)
  Raise $ 2,000/yr  →  ~$3,500/mo mortgage  (P=48%, mean net = $1,805)
  Raise $ 3,000/yr  →  ~$3,500/mo mortgage  (P=52%, mean net = $2,805)
  Raise $ 4,000/yr  →  ~$3,600/mo mortgage  (P=51%, mean net = $2,605)
  Raise $ 5,000/yr  →  ~$3,700/mo mortgage  (P=50%, mean net = $2,405)
  Raise $ 6,000/yr  →  ~$3,800/mo mortgage  (P=50%, mean net = $2,205)
  Raise $ 7,000/yr  →  ~$3,900/mo mortgage  (P=49%, mean net = $2,005)
  Raise $ 8,000/yr  →  ~$4,000/mo mortgage  (P=48%, mean net = $1,805)
  Raise $ 9,000/yr  →  ~$4,000/mo mortgage  (P=52%, mean net = $2,805)
  Raise $10,000/yr  →  ~$4,100/mo mortgage  (P=51%, mean net = $2,605)
  Raise $11,000/yr  →  ~$4,200/mo mortgage  (P=50%, mean net = $2,405)
  Raise $12,000/yr  →  ~$4,300/mo mortgage  (P=50%, mean net = $2,205)
  Raise $13,000/yr  →  ~$4,400/mo mortgage  (P=49%, mean net = $2,005